In [ ]:
# Required imports
import requests
import time

# Multiple public Overpass API instances to try in order 
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]

def query_overpass(ql: str, timeout: int = 30) -> dict:
    """
    Try each Overpass mirror in turn.
    Uses Accept: */* — never application/json, which causes 406 on some mirrors.
    """
    headers = {
        "User-Agent": "road-lane-checker/1.0 (research project)",
        "Accept": "*/*",                        
        "Content-Type": "application/x-www-form-urlencoded",
    }

    last_error = None
    for endpoint in OVERPASS_ENDPOINTS:
        try:
            print(f"[Overpass] Trying {endpoint} ...")
            r = requests.post(
                endpoint,
                data={"data": ql},             # form-encoded — most compatible
                headers=headers,
                timeout=timeout,
            )
            if r.status_code == 429:
                print(f"  → Rate limited. Waiting 5 s ...")
                time.sleep(5)
                r = requests.post(endpoint, data={"data": ql},
                                  headers=headers, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            print(f"  → HTTP {r.status_code}. Trying next mirror.")
            last_error = f"HTTP {r.status_code}"
        except requests.exceptions.Timeout:
            print(f"  → Timeout. Trying next mirror.")
            last_error = "Timeout"
        except requests.exceptions.ConnectionError as e:
            print(f"  → Connection error: {e}. Trying next mirror.")
            last_error = str(e)

    raise RuntimeError(
        f"All Overpass mirrors failed. Last error: {last_error}\n"
        "Check your internet connection or try a VPN — "
        "some mirrors are blocked in certain regions."
    )


# ── 2. Road check using the query function ────────────────────────────
def check_overpass_directly(lat: float, lon: float, radius_m: int = 200) -> dict:
    ql = f"""
[out:json][timeout:25];
(
  way["highway"](around:{radius_m},{lat},{lon});
);
out tags;
"""
    data = query_overpass(ql)
    elements = data.get("elements", [])

    if not elements:
        return {"found": False, "message": f"No highway ways within {radius_m}m of ({lat}, {lon})"}

    roads = []
    for el in elements[:5]:
        tags = el.get("tags", {})
        roads.append({
            "osm_id":  el["id"],
            "name":    tags.get("name", "unnamed"),
            "highway": tags.get("highway"),
            "lanes":   tags.get("lanes"),
            "oneway":  tags.get("oneway"),
        })

    return {"found": True, "road_count": len(elements), "sample_roads": roads}

In [15]:
diag = check_overpass_directly(lat=22.4380919, lon=114.0653398)
print(diag)

[Overpass] Trying https://overpass-api.de/api/interpreter ...
{'found': True, 'road_count': 44, 'sample_roads': [{'osm_id': 111972346, 'name': '錦莆路 Kam Po Road', 'highway': 'residential', 'lanes': None, 'oneway': 'yes'}, {'osm_id': 160073817, 'name': '錦匯路 Kam Wui Road', 'highway': 'residential', 'lanes': '1', 'oneway': None}, {'osm_id': 163214173, 'name': 'unnamed', 'highway': 'footway', 'lanes': None, 'oneway': None}, {'osm_id': 245642588, 'name': 'unnamed', 'highway': 'service', 'lanes': None, 'oneway': None}, {'osm_id': 245642589, 'name': 'unnamed', 'highway': 'service', 'lanes': None, 'oneway': None}]}
